<a href="https://colab.research.google.com/github/N-Uma/Machine-Learning-and-Big-Data/blob/main/Model_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from pyspark.sql import SparkSession
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.regression import GBTRegressionModel
from pyspark.sql.types import *
import numpy as np

path = "/content/drive/MyDrive/NYC_Taxi_Yellow_2012"
BASE_PATH = path


if "spark" in locals():
    try:
        spark.stop()
    except Exception as err:
        print(f"Warning: Spark session could not be stopped: {err}")


spark = (
    SparkSession.builder
    .appName("NYC Taxi Prediction")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .getOrCreate()
)

train_df = spark.read.parquet(f"{BASE_PATH}/data/final/train_2012")
val_df   = spark.read.parquet(f"{BASE_PATH}/data/final/val_2012")
test_df  = spark.read.parquet(f"{BASE_PATH}/data/final/test_2012")


train_count = train_df.count()
val_count   = val_df.count()
test_count  = test_df.count()
total_count = train_count + val_count + test_count

print(f"Train (Jan–Oct): {train_count:,} rows ({train_count / total_count * 100:.1f}")
print(f"Val (Nov):       {val_count:,} rows")
print(f"Test (Dec):      {test_count:,} rows")

Train (Jan–Oct): 140,517,318 rows (83.3
Val (Nov):       13,663,801 rows
Test (Dec):      14,572,953 rows


In [4]:
from pyspark.ml.regression import RandomForestRegressor


rf_model = RandomForestRegressor(
    featuresCol="features",
    labelCol="fare_amount",
    numTrees=3,
    maxDepth=3,
    subsamplingRate=0.7,
    seed=42
)


best_model = rf_model.fit(train_df)

In [5]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import Row

print("\nDISTRIBUTED REGRESSION METRICS\n")


test_preds = best_model.transform(test_df).cache()
test_preds.count()


evaluators = {
    "RMSE": RegressionEvaluator(
        labelCol="fare_amount",
        predictionCol="prediction",
        metricName="rmse"
    ),
    "MSE": RegressionEvaluator(
        labelCol="fare_amount",
        predictionCol="prediction",
        metricName="mse"
    ),
    "MAE": RegressionEvaluator(
        labelCol="fare_amount",
        predictionCol="prediction",
        metricName="mae"
    ),
    "R2": RegressionEvaluator(
        labelCol="fare_amount",
        predictionCol="prediction",
        metricName="r2"
    )
}


metrics = {}

for metric_name, evaluator in evaluators.items():
    value = evaluator.evaluate(test_preds)
    metrics[metric_name] = __builtins__.round(value, 4)
    print(f"{metric_name:<6}: {metrics[metric_name]}")


summary_df = spark.createDataFrame([Row(**metrics)])
print("\nSUMMARY TABLE:")
summary_df.show(truncate=False)


test_preds.unpersist()


DISTRIBUTED REGRESSION METRICS

RMSE  : 6.8305
MSE   : 46.6561
MAE   : 4.377
R2    : 0.4987

SUMMARY TABLE:
+------+-------+-----+------+
|RMSE  |MSE    |MAE  |R2    |
+------+-------+-----+------+
|6.8305|46.6561|4.377|0.4987|
+------+-------+-----+------+



DataFrame[features: vector, fare_amount: double, prediction: double]

In [6]:
from pyspark.sql.functions import col, avg, lit

print("\nTEMPORAL SPLIT ANALYSIS")
print("Temporal ordering confirmed:")
print("  Train: Jan–Oct")
print("  Val:   Nov")
print("  Test:  Dec")


monthly_perf = (
    test_preds
    .withColumn("pickup_month", lit("2015-12"))
    .groupBy("pickup_month")
    .agg(
        avg(col("prediction") - col("fare_amount")).alias("mean_error"),
        avg(col("fare_amount")).alias("avg_fare")
    )
)


monthly_perf.show(truncate=False)


TEMPORAL SPLIT ANALYSIS
Temporal ordering confirmed:
  Train: Jan–Oct
  Val:   Nov
  Test:  Dec
+------------+-------------------+-----------------+
|pickup_month|mean_error         |avg_fare         |
+------------+-------------------+-----------------+
|2015-12     |-1.5520987343056138|12.23140735237393|
+------------+-------------------+-----------------+



In [7]:
from pyspark.sql.functions import col, when
from pyspark.ml.evaluation import RegressionEvaluator
import numpy as np

print("\n5-FOLD CROSS-VALIDATION (APPROX. STRATIFIED)")


fare_buckets = (
    test_preds
    .withColumn(
        "fare_bucket",
        when(col("fare_amount") < 10, "low")
        .when(col("fare_amount") < 25, "medium")
        .otherwise("high")
    )
)


print("Fare distribution:")
(
    fare_buckets
    .groupBy("fare_bucket")
    .count()
    .orderBy("fare_bucket")
    .show()
)


evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)

cv_scores = []


for fold_id in range(5):

    fold_df = (
        fare_buckets
        .sample(
            withReplacement=False,
            fraction=0.2,
            seed=fold_id
        )
        .select("features", "fare_amount")
        .cache()
    )


    fold_df.count()

    fold_predictions = best_model.transform(fold_df)

    rmse = evaluator.evaluate(fold_predictions)
    cv_scores.append(rmse)

    fold_df.unpersist()


mean_rmse = np.mean(cv_scores)
std_rmse = np.std(cv_scores)

print(f"Mean RMSE: ${mean_rmse:.2f} ± {std_rmse:.2f}")
print("Fold RMSEs:", [f"{score:.2f}" for score in cv_scores])


5-FOLD CROSS-VALIDATION (APPROX. STRATIFIED)
Fare distribution:
+-----------+-------+
|fare_bucket|  count|
+-----------+-------+
|       high|1191475|
|        low|7767230|
|     medium|5614248|
+-----------+-------+

Mean RMSE: $6.83 ± 0.01
Fold RMSEs: ['6.84', '6.84', '6.82', '6.83', '6.81']


In [8]:
from pyspark.ml.evaluation import RegressionEvaluator
import numpy as np

print("\nBOOTSTRAP CONFIDENCE INTERVALS")


test_preds_cached = (
    best_model
    .transform(test_df.coalesce(10))
    .select("prediction", "fare_amount")
    .cache()
)


test_preds_cached.count()


evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)


def bootstrap_rmse_ci(n_bootstrap=50):
    rmse_scores = []

    for seed in range(n_bootstrap):
        bootstrap_sample = (
            test_preds_cached
            .sample(
                withReplacement=True,
                fraction=0.8,
                seed=seed
            )
        )

        rmse = evaluator.evaluate(bootstrap_sample)
        rmse_scores.append(rmse)

    rmse_scores = np.array(sorted(rmse_scores))

    lower_idx = int(0.025 * len(rmse_scores))
    upper_idx = int(0.975 * len(rmse_scores))

    mean_rmse = rmse_scores.mean()
    ci_lower = rmse_scores[lower_idx]
    ci_upper = rmse_scores[upper_idx]

    return mean_rmse, ci_lower, ci_upper


mean_rmse, ci_low, ci_high = bootstrap_rmse_ci(n_bootstrap=50)

print("RMSE Bootstrap Results:")
print(f"  Mean RMSE: ${mean_rmse:.2f}")
print(f"  95% CI:    ${ci_low:.2f} – ${ci_high:.2f}")
print(f"  CI Width:  ${ci_high - ci_low:.2f} (narrow = stable model)")


test_preds_cached.unpersist()


BOOTSTRAP CONFIDENCE INTERVALS
RMSE Bootstrap Results:
  Mean RMSE: $6.83
  95% CI:    $6.82 – $6.85
  CI Width:  $0.03 (narrow = stable model)


DataFrame[prediction: double, fare_amount: double]

In [9]:
from pyspark.sql.functions import col, avg, abs as sql_abs, count, when

print("\nBUSINESS METRICS ALIGNMENT")


test_with_profit = (
    test_preds
    .select("fare_amount", "prediction")
    .withColumn("driver_profit", col("fare_amount"))
    .withColumn("predicted_profit", col("prediction"))
)

business_metrics_row = (
    test_with_profit
    .agg(
        avg(col("driver_profit")).alias("avg_driver_profit"),
        avg(col("predicted_profit")).alias("avg_pred_profit"),
        avg(sql_abs(col("driver_profit") - col("predicted_profit")))
            .alias("profit_forecast_error"),
        count(when(col("predicted_profit") > 0, True))
            .alias("profitable_trips_pred"),
        count(when(col("driver_profit") > 0, True))
            .alias("profitable_trips_actual")
    )
    .first()
)


print("NYC Taxi Driver Economics:")
print(f"Actual avg profit / trip:     ${business_metrics_row['avg_driver_profit']:.2f}")
print(f"Predicted avg profit / trip:  ${business_metrics_row['avg_pred_profit']:.2f}")
print(f"Profit forecast error:        ${business_metrics_row['profit_forecast_error']:.2f}")
print(f"Profitable trips (actual):    {business_metrics_row['profitable_trips_actual']:,}")
print(f"Profitable trips (predicted): {business_metrics_row['profitable_trips_pred']:,}")


BUSINESS METRICS ALIGNMENT
NYC Taxi Driver Economics:
Actual avg profit / trip:     $12.23
Predicted avg profit / trip:  $10.68
Profit forecast error:        $4.38
Profitable trips (actual):    14,572,953
Profitable trips (predicted): 14,572,953
